# W28 · 阶段项目：仿真机器人自主导航系统集成

> 把 W21–W27 的每个零件——URDF/xacro、Gazebo 传感器、ros2_control、
> Nav2、RL 策略节点——装配成一个**端到端自主导航系统**，并用工程化方法评估它。
> 这一讲没有新知识点，考的是系统集成与交付能力（中级工程师的核心竞争力）。

## 学习目标

1. 设计并讲清一个分层自主导航系统的架构与数据流；
2. 用 launch 编排整套系统（一键启动）；
3. 建立**可复现的评估协议**：成功率、路径效率（SPL）、碰撞次数；
4. 完成一份包含录屏与故障排查记录的工程交付；
5. 为阶段四（Isaac Lab / Capstone）的可选方向积累系统资产。

## ⚠️ 运行前提

集成与验收需 ROS2 Jazzy + Gazebo + Nav2 全栈环境；SPL 评估指标演示为纯 Python，本机真实执行。

## 1. 项目定义与验收标准

**题目**：在 Gazebo 室内场景中，差速机器人从起点自主导航到用户指定的目标点；
导航过程可由你封装的 RL 局部策略接管/增强（对应 W27 的集成方式 1 或 2）。

**验收标准（Definition of Done）**：

- [ ] 一条 `ros2 launch` 命令拉起全系统（Gazebo + 机器人 + 桥 + 控制 + Nav2/RL）；
- [ ] RViz 下发目标点后机器人无碰撞到达，连续 10 次成功率 ≥ 80%；
- [ ] 每次运行自动记录评估指标（见第 4 节）；
- [ ] 交付：仓库（含 launch/配置/README）、评估报告、≥ 2 分钟录屏。

**建议里程碑（两周节奏）**：

| 里程碑 | 内容 | 出口标准 |
|--------|------|----------|
| M0 | 机器人模型收尾 | URDF+xacro 通过 check_urdf；RViz 显示正确 |
| M1 | 仿真与传感 | Gazebo 中 spawn；/scan、/odom、TF 树完整 |
| M2 | 控制链 | ros2_control 全 active；键盘/cmd_vel 驱动正常；odom 合理 |
| M3 | 导航基线 | SLAM 建图 → AMCL + Nav2 跑通 10 次到点 |
| M4 | RL 增强 | 策略节点接入（替换 controller 或 twist_mux 仲裁），对比实验 |
| M5 | 评估与交付 | 评估脚本、报告、录屏 |

## 2. 参考架构与文件树

```
my_nav_robot/
├── my_robot_description/        # M0
│   ├── urdf/diffbot.urdf.xacro  #   底盘+轮子+lidar+imu（W23/W24 标签齐全）
│   └── launch/description.launch.py
├── my_robot_simulation/         # M1
│   ├── worlds/indoor.sdf
│   ├── config/bridge.yaml       #   ros_gz 桥接清单（W24）
│   └── launch/sim.launch.py
├── my_robot_control/            # M2
│   ├── config/controllers.yaml  #   diff_drive_controller 链（W25）
│   └── launch/control.launch.py
├── my_robot_navigation/         # M3
│   ├── config/nav2_params.yaml  #   从 nav2_bringup 模板改（W26）
│   ├── maps/indoor.yaml
│   └── launch/nav.launch.py
├── my_rl_policy/                # M4
│   ├── my_rl_policy/rl_policy_node.py   # W27 模板
│   └── policies/ppo_local_planner.zip
└── my_robot_bringup/            # M5：一键启动
    └── launch/bringup.launch.py #   include 以上所有 launch
```

**数据流自检清单**（排错时沿此逐段 `ros2 topic echo/hz`）：

`/clock → /scan → TF(map→odom→base_link→lidar) → costmap → /plan → /cmd_vel
→ diff_drive_controller → Gazebo 关节 → /odom`。

## 3. 一键启动：bringup.launch.py 骨架

```python
from launch import LaunchDescription
from launch.actions import IncludeLaunchDescription
from launch.launch_description_sources import PythonLaunchDescriptionSource
from launch.substitutions import PathJoinSubstitution
from launch_ros.substitutions import FindPackageShare


def include(pkg, name, **kwargs):
    return IncludeLaunchDescription(
        PythonLaunchDescriptionSource(
            PathJoinSubstitution([FindPackageShare(pkg), "launch", name])),
        launch_arguments=kwargs.items(),
    )


def generate_launch_description():
    return LaunchDescription([
        include("my_robot_simulation", "sim.launch.py", world="indoor.sdf"),
        include("my_robot_control", "control.launch.py"),
        include("my_robot_navigation", "nav.launch.py", use_sim_time="true"),
        # M4 之后取消注释：策略节点与 Nav2 controller 二选一/或 twist_mux
        # include("my_rl_policy", "policy.launch.py", policy_path="ppo_local_planner.zip"),
    ])
```

**工程要点**：所有节点 `use_sim_time:=true`（W24 的教训）；启动顺序用
`RegisterEventHandler`/`TimerAction` 处理依赖（Gazebo 没起来就 spawn 会失败）；
失败节点的日志级别在 bringup 里统一设置，方便录屏时观众看懂。

## 4. 评估协议：成功率之外的 SPL

只报「成功率」会鼓励「绕远路但保守」的策略。导航社区的标准指标
**SPL（Success weighted by Path Length）**：

$$
\mathrm{SPL} = \frac{1}{N}\sum_{i=1}^{N} S_i \cdot \frac{\ell_i}{\max(p_i,\ \ell_i)}
$$

$S_i \in \{0,1\}$ 是否成功，$\ell_i$ 起终点最短路径长度，$p_i$ 实际路径长度。
SPL ≤ 成功率，且路径越接近最优越接近 1。下面实现并理解它（本机执行）：

In [1]:
"""SPL 评估指标：实现 + 用模拟数据理解其性质（纯 Python，本机执行）。"""
import numpy as np


def path_length(waypoints: np.ndarray) -> float:
    """折线轨迹总长度。waypoints: (T, 2)。"""
    return float(np.sum(np.linalg.norm(np.diff(waypoints, axis=0), axis=1)))


def spl(successes, shortest_paths, actual_paths) -> float:
    successes, shortest_paths, actual_paths = map(np.asarray, (successes, shortest_paths, actual_paths))
    terms = successes * shortest_paths / np.maximum(actual_paths, shortest_paths)
    return float(np.mean(terms))


# --- 模拟 6 次导航试验 ---
rng = np.random.default_rng(0)
records = []
for i in range(6):
    start, goal = np.array([0.0, 0.0]), np.array([5.0, 0.0])
    shortest = float(np.linalg.norm(goal - start))          # ℓ = 5.0
    detour = rng.uniform(0, 1.5)                             # 绕路程度
    mid = np.array([2.5, detour])
    traj = np.array([start, mid, goal])
    success = detour < 1.2                                    # 绕太远的那次撞了
    records.append((success, shortest, path_length(traj)))

s = [r[0] for r in records]
ell = [r[1] for r in records]
p = [r[2] for r in records]

print(f"{'trial':>5} {'success':>7} {'ℓ':>5} {'p':>6} {'单项 SPL':>8}")
for i, (si, li, pi) in enumerate(records):
    print(f"{i:>5} {str(si):>7} {li:5.2f} {pi:6.2f} {si * li / max(pi, li):8.3f}")
print(f"\n成功率 = {np.mean(s):.2f},  SPL = {spl(s, ell, p):.3f}")
print("结论：SPL ≤ 成功率；绕路越多 SPL 越低——它同时惩罚失败与低效。")

trial success     ℓ      p   单项 SPL
    0    True  5.00   5.35    0.934
    1    True  5.00   5.07    0.987
    2    True  5.00   5.00    1.000
    3    True  5.00   5.00    1.000
    4   False  5.00   5.56    0.000
    5   False  5.00   5.70    0.000

成功率 = 0.67,  SPL = 0.653
结论：SPL ≤ 成功率；绕路越多 SPL 越低——它同时惩罚失败与低效。


**评估协议模板**（写进你的报告）：

1. **场景**：同一地图、固定 5 组起终点对（覆盖近/中/远 + 必经窄通道）；
2. **重复**：每组跑 4 次，共 20 次试验；记录成功、轨迹、用时、最小障碍距离；
3. **对比**：Nav2 基线 vs RL 增强，同样 20 次；
4. **报告**：成功率、SPL、平均用时、碰撞次数，附典型成功/失败轨迹图。

## 5. 故障排查清单（系统集成期的高频坑）

| 症状 | 第一嫌疑 | 验证手段 |
|------|----------|----------|
| RViz 里机器人不动但 cmd_vel 有值 | 控制器没 active / cmd_vel 话题名不匹配 | `ros2 control list_controllers`、`ros2 topic info -v` |
| TF 报「extrapolation into future」 | `use_sim_time` 漏设 / `/clock` 没桥 | `ros2 param get <node> use_sim_time` |
| 代价地图全黑/全灰 | `/scan` 的 frame_id 不在 TF 树上 | `ros2 run tf2_ros tf2_echo map <scan frame>` |
| AMCL 定位乱跳 | odom 漂移大 / lidar 与地图对不上 | 先静止看 `/amcl_pose` 是否稳定 |
| 规划路径穿墙 | `robot_radius` 或膨胀参数太小 / 地图分辨率错 | RViz 对比 costmap 与实际几何 |
| 策略节点输出僵住 | watchdog 触发（观测话题名错） | 节点日志的 warn 节流输出 |
| 到点附近振荡 | controller `sim_time` 太短 / xy_goal_tolerance 太紧 | 逐个放宽复现 |

把你自己踩过的坑追加进这张表——这就是 `journal/troubleshooting.md` 的意义。

## ✏️ 练习

### 练习 1（★，约 20 分钟）：项目计划书

写一页项目计划：架构图（自己画）、里程碑排期（对齐你的日历）、
风险清单（至少 3 条，含缓解措施）。交付：`journal/phase3_project_plan.md`。

### 练习 2（★★，约 90 分钟，可拆两周）：M0–M2

完成机器人模型、Gazebo 集成与 ros2_control 控制链。
交付：三个包 + 每个里程碑的出口标准证据（check_urdf 输出、TF 树图、
`ros2 control list_controllers` 全 active）。

### 练习 3（★★，约 60 分钟）：M3 导航基线

SLAM 建图 → 保存地图 → AMCL + Nav2 连续 10 次到点，记录成功率。
交付：地图文件、nav2_params.yaml 与你改动的每个参数的理由、10 次试验记录表。

### 练习 4（★★★，约 120 分钟）：M4–M5 RL 增强与对比评估

用 W27 模板把局部策略接入（替换 controller_server 或 twist_mux 仲裁），
按第 4 节评估协议跑 Nav2 基线 vs RL 增强各 20 次，用本讲 SPL 函数计算指标。
交付：对比表格 + 典型轨迹图 + ≥ 2 分钟录屏 + 一段「RL 相比经典方法强在哪、弱在哪」的诚实分析。

## 参考答案

<details>
<summary>参考答案</summary>

**练习 1**：风险清单示例——(a) Gazebo 与 ros2_control 版本兼容（缓解：先用官方
ros2_control_demos 的 gz 例程打底）；(b) 策略推理延迟超控制周期（缓解：先用 50 Hz 以下频率，
Profiler 测 P99）；(c) 地图质量差导致 AMCL 不稳（缓解：建图时慢速多走回环，必要时手工修图）。

**练习 2**：出口标准见正文表格；常见卡点——lidar 的 `<gazebo reference>` 必须指向
link 名而非 joint 名；`<ros2_control>` 的 joint 名要与 URDF 完全一致（xacro 变量展开后核对）。

**练习 3**：参数改动理由的写法示例——「`inflation_radius: 0.55→0.40`：机器人需通过
0.8 m 宽门，0.55 的膨胀使通道中心代价超阈值导致规划失败」；每条理由配一次前后对比实验。

**练习 4**：诚实分析的方向——RL 策略在窄通道/动态障碍上的平均用时通常更低
（学到了「预判式」减速），但失败模式更难解释、对训练分布外的场景（全新家具布局）
退化比 Nav2 更陡；工程上 twist_mux 仲裁（Nav2 主导 + RL 安全层）往往是
成功率与可解释性的最佳折中。SPL 对比务必同场景同种子区间，避免「挑着报」。
</details>

## 延伸阅读

- [Nav2 官方教程全集](https://docs.nav2.org/tutorials/index.html)（含自定义控制器/规划器插件教程——M4 的进阶路线）
- [ros2_control_demos](https://github.com/ros-controls/ros2_control_demos)（M0–M2 的可靠起点）
- [SPL 指标出处：Anderson et al., An Evaluation of Visual Navigation Agents (CVPR 2018)](https://arxiv.org/abs/1807.06757)
- [Articulated Robotics（YouTube）](https://www.youtube.com/@ArticulatedRobotics)：ROS2 + 真实机器人全流程，本项目 M0–M3 的绝佳视频参照
- 阶段四预告：M10–M12 选修 Isaac Lab/OpenUSD，Capstone 可把本项目升级为
  「Isaac Lab 训练 → 域随机化 → Gazebo/ROS2 部署」的完整 Sim2Real 流水线。